# Self Analysis

Interactive version of `src/compare_self.py` -- same underlying data and cluster models, but
with inline charts and a percentile table you can filter/sort/re-slice on the fly, instead of
a fixed printed report. Run `00_pull_data.ipynb` and `src/build_clusters.py` first if you
haven't already.

In [ ]:
import sys
sys.path.append('../src')

import joblib
import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import percentileofscore

from feature_config import ALL_FEATURE_FIELDS, VALID_ROLES
from export_dashboard_data import resolve_puuid_from_match_ids

FEATURE_COLUMNS = [f.name for f in ALL_FEATURE_FIELDS]

RIOT_ID = 'river3#na1'
REFERENCE_TIER = 'CHALLENGER'
FOCUS_ROLE = 'BOTTOM'  # one of VALID_ROLES

df = pd.read_parquet('../data/processed/participant_features.parquet')
puuid = resolve_puuid_from_match_ids('../data/raw/match_ids.ndjson', RIOT_ID)
my_df = df[df['puuid'] == puuid]
print(f'{len(my_df)} games found for {RIOT_ID}')
my_df['teamPosition'].value_counts()

## Percentile table

Every candidate feature (not just the dashboard's curated subset), your average, the
reference-tier average, and your percentile -- styled so outliers jump out. Change
`FOCUS_ROLE` above and re-run to look at a different role.

In [ ]:
my_role_df = my_df[my_df['teamPosition'] == FOCUS_ROLE]
reference_df = df[(df['tier'] == REFERENCE_TIER) & (df['teamPosition'] == FOCUS_ROLE)]

rows = []
for col in FEATURE_COLUMNS:
    your_avg = my_role_df[col].mean()
    ref_series = reference_df[col].dropna()
    percentile = percentileofscore(ref_series, your_avg, kind='mean') if len(ref_series) and pd.notna(your_avg) else None
    rows.append({'feature': col, 'your_avg': your_avg, 'reference_avg': ref_series.mean(), 'percentile': percentile})

percentile_table = pd.DataFrame(rows).sort_values('percentile')
percentile_table.style.background_gradient(subset=['percentile'], cmap='RdYlGn', vmin=0, vmax=100).format({'your_avg': '{:.2f}', 'reference_avg': '{:.2f}', 'percentile': '{:.0f}'})

## Percentile bar chart

Same idea as the dashboard's diverging bar chart, quick-and-dirty in matplotlib for
whatever feature subset you want to eyeball right now.

In [ ]:
FEATURES_TO_PLOT = ['damage_per_minute', 'gold_earned_per_min', 'vision_score_per_minute', 'kda',
                     'kill_participation', 'cs_per_min', 'deaths_per_min', 'damage_taken_per_min']

plot_df = percentile_table[percentile_table['feature'].isin(FEATURES_TO_PLOT)].set_index('feature').loc[FEATURES_TO_PLOT]
colors = ['#2a78d6' if v >= 50 else '#e34948' for v in plot_df['percentile']]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(plot_df.index, plot_df['percentile'] - 50, left=50, color=colors)
ax.axvline(50, color='gray', linewidth=1)
ax.set_xlim(0, 100)
ax.set_xlabel(f'Percentile vs. {REFERENCE_TIER.title()} {FOCUS_ROLE.title()}')
ax.invert_yaxis()
fig.tight_layout()
plt.show()

## Your cluster distribution

Which learned archetype(s) your games in this role fall into, and how those archetypes'
population win rates compare.

In [ ]:
artifact = joblib.load(f'../models/{FOCUS_ROLE.lower()}_cluster.joblib')
role_pop_df = pd.read_parquet('../data/processed/participant_clusters.parquet')
role_pop_df = role_pop_df[role_pop_df['teamPosition'] == FOCUS_ROLE]

matrix = my_role_df[FEATURE_COLUMNS].fillna(role_pop_df[FEATURE_COLUMNS].median(numeric_only=True))
scaled = artifact['scaler'].transform(matrix)
reduced = artifact['pca'].transform(scaled)
my_clusters = artifact['kmeans'].predict(reduced)

cluster_win_rates = role_pop_df.groupby('cluster')['win'].mean()
your_counts = pd.Series(my_clusters).value_counts().sort_index()

summary = pd.DataFrame({'population_win_rate': cluster_win_rates, 'your_games': your_counts}).fillna(0)
summary['your_games'] = summary['your_games'].astype(int)
summary

## Stat trend over your last N games

Not in the static dashboard -- a per-game view to spot whether you're trending up or down on
a given stat, most recent game last.

In [ ]:
import json

def game_creation_by_match_id(matches_path, match_ids):
    creation = {}
    with open(matches_path, encoding='utf-8') as f:
        for line in f:
            row = json.loads(line)
            match_id = row.get('metadata', {}).get('matchId')
            if match_id in match_ids:
                creation[match_id] = row.get('info', {}).get('gameCreation')
    return creation

TREND_STAT = 'kda'

trend_df = my_role_df.copy()
creation = game_creation_by_match_id('../data/raw/matches.ndjson', set(trend_df['matchId']))
trend_df['gameCreation'] = trend_df['matchId'].map(creation)
trend_df = trend_df.sort_values('gameCreation')

fig, ax = plt.subplots(figsize=(9, 3.5))
colors = ['#2a78d6' if w else '#e34948' for w in trend_df['win']]
ax.scatter(range(len(trend_df)), trend_df[TREND_STAT], c=colors)
ax.axhline(reference_df[TREND_STAT].mean(), color='gray', linestyle='--', linewidth=1, label=f'{REFERENCE_TIER.title()} avg')
ax.set_xlabel('Game (oldest -> newest)')
ax.set_ylabel(TREND_STAT)
ax.legend()
fig.tight_layout()
plt.show()